# Cuaderno ML 1: Segmentación de ciclos de vida de juegos con K-Means

**Institución Universitaria Pascual Bravo**
**Asignatura:** Gestión de Datos Masivos
**Docente:** Genry Gutiérrez
**Proyecto:** Steam Game Intelligence
**Infraestructura:** Databricks Lakehouse, Unity Catalog, Delta Lake y Spark MLlib
**Fuente:** capa Gold (`game_growth`, `dim_game`, `twitch_steam_relationship`)

---

## Pregunta de negocio

> **¿Qué ciclos de vida típicos tienen los juegos de Steam y qué exposición en Twitch tiene cada uno?**

El análisis descriptivo de Gold mostró una correlación prácticamente nula (−0,0032) entre las horas vistas en Twitch y el crecimiento posterior en Steam. Una correlación global mezcla juegos muy distintos, así que primero se agrupan los juegos según la **forma de su ciclo de vida** y después se compara la exposición en Twitch entre esos grupos.

## Objetivos

1. Construir desde Gold una tabla con **una fila por juego** que describa su ciclo de vida.
2. Preparar las variables para un modelo basado en distancias.
3. Entrenar **K-Means** y elegir el número de grupos con la silueta y el método del codo.
4. Interpretar los arquetipos y comparar su exposición en Twitch.

---
## 1. ¿Por qué un modelo no supervisado?

| Criterio | No supervisado (este cuaderno) | Supervisado (Cuaderno ML 2) |
| :--- | :--- | :--- |
| **Variable objetivo** | No existe: nadie etiquetó a los juegos como "longevo" o "de caída rápida". | Sí existe: si el juego creció o no el mes siguiente. |
| **Pregunta** | ¿Qué tipos de ciclo de vida existen? | ¿Este juego crecerá el próximo mes? |

### K-Means

K-Means divide los juegos en $k$ grupos. Cada juego pertenece al grupo con el **centroide** más cercano, y el algoritmo minimiza la suma de distancias al cuadrado de cada punto a su centroide (inercia):

$$J = \sum_{j=1}^{k} \sum_{\mathbf{x}_i \in C_j} \lVert \mathbf{x}_i - \boldsymbol{\mu}_j \rVert^2$$

1. Elige $k$ centroides iniciales.
2. Asigna cada juego al centroide más cercano.
3. Recalcula cada centroide como el promedio de sus juegos.
4. Repite 2 y 3 hasta que los centroides dejan de moverse.

### Decisiones de diseño

1. **Una fila por juego.** Si se agruparan filas mensuales, un mismo juego quedaría repartido en varios grupos.
2. **Solo variables de comportamiento.** Género, precio, reseñas y Twitch se usan **después**, para describir los grupos, no para formarlos.
3. **Logaritmo en el número de jugadores.** Pocos juegos tienen cientos de miles de jugadores y miles tienen menos de 100; sin `log1p` los gigantes formarían su propio grupo.
4. **Estandarización.** Las variables están en unidades distintas, así que `StandardScaler` las lleva a media 0 y desviación 1.

---
## 2. Configuración

In [ ]:
%pip install plotly==5.24.1 --quiet

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pyspark.ml import Pipeline
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import PCA, StandardScaler, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.sql import Window
from pyspark.sql import functions as F

GOLD = "steam_game_intelligence.gold"

MIN_MONTHS = 12
MIN_PEAK = 20
K_VALUES = range(3, 9)
SEED = 42

TWITCH_START = "2016-01-01"
TWITCH_END = "2024-09-01"

---
## 3. Carga de la capa Gold

In [ ]:
game_growth = spark.table(f"{GOLD}.game_growth")
dim_game = spark.table(f"{GOLD}.dim_game")
twitch = spark.table(f"{GOLD}.twitch_steam_relationship")

for name, df in [("game_growth", game_growth), ("dim_game", dim_game), ("twitch_steam_relationship", twitch)]:
    print(f"{name}: {df.count():,} filas | {df.select('app_id').distinct().count():,} juegos")

display(game_growth.orderBy("app_id", "month_start").limit(10))

---
## 4. Variables por juego

Se parte de `gold.game_growth` (una fila por juego y mes) y se resume la historia de cada juego en siete variables:

| Variable | Qué mide | Cálculo |
|---|---|---|
| `log_peak_players` | Tamaño de la comunidad | `log1p(máximo de avg_players)` |
| `months_observed` | Longevidad | Número de meses con datos |
| `pct_time_to_peak` | Momento del pico | 0 = pico al inicio, 1 = pico al final |
| `retention_ratio` | Retención | Promedio de los últimos 3 meses ÷ pico |
| `trend_slope_yearly` | Tendencia | Pendiente anual de `log1p(avg_players)` |
| `volatility` | Estabilidad | Desviación estándar del cambio mensual en logaritmo |
| `log_peak_to_avg` | Picos dentro del mes | `log(promedio de peak_to_avg_ratio)` |

La volatilidad usa diferencias de logaritmos y no `mom_change_pct`, porque ese porcentaje explota cuando el mes anterior tiene casi cero jugadores.

In [ ]:
numeric_cols = ["avg_players", "prev_avg_players", "peak_to_avg_ratio", "mom_change_pct"]

monthly = (
    game_growth
    .filter(F.col("avg_players").isNotNull())
    .select("app_id", "name", "month_start", *[F.col(c).cast("double").alias(c) for c in numeric_cols])
    .withColumn("month_idx", F.year("month_start") * 12 + F.month("month_start"))
    .withColumn("log_players", F.log1p("avg_players"))
    .withColumn("log_diff", F.when(F.col("mom_change_pct").isNotNull(), F.log1p("avg_players") - F.log1p("prev_avg_players")))
    .withColumn("rn_desc", F.row_number().over(Window.partitionBy("app_id").orderBy(F.desc("month_start"))))
)

per_game = (
    monthly.groupBy("app_id")
    .agg(
        F.max("name").alias("name"),
        F.count("*").alias("months_observed"),
        F.min("month_start").alias("first_month"),
        F.max("month_start").alias("last_month"),
        F.min("month_idx").alias("first_idx"),
        F.max("month_idx").alias("last_idx"),
        F.max("avg_players").alias("peak_players"),
        F.max_by("month_idx", "avg_players").alias("peak_idx"),
        F.avg(F.when(F.col("rn_desc") <= 3, F.col("avg_players"))).alias("avg_last3"),
        F.expr("regr_slope(log_players, month_idx)").alias("slope_monthly"),
        F.stddev("log_diff").alias("volatility"),
        F.avg("peak_to_avg_ratio").alias("mean_peak_to_avg"),
    )
    .cache()
)

print(f"Juegos con historial: {per_game.count():,}")

### 4.1 Población del modelo

Se aplican dos filtros antes de agrupar:

- **Historial mínimo de 12 meses:** con menos meses no se puede hablar de ciclo de vida.
- **Pico mínimo de 20 jugadores:** en juegos casi vacíos los cambios son ruido.

La línea roja de cada histograma marca el umbral elegido.

In [ ]:
hist_pd = per_game.select("months_observed", "peak_players").toPandas()

fig = make_subplots(rows=1, cols=2, subplot_titles=("Meses observados por juego", "Pico de jugadores por juego (log10)"))
fig.add_trace(go.Histogram(x=hist_pd["months_observed"], nbinsx=60), row=1, col=1)
fig.add_trace(go.Histogram(x=np.log10(hist_pd["peak_players"] + 1), nbinsx=60), row=1, col=2)
fig.add_vline(x=MIN_MONTHS, line_dash="dash", line_color="red", row=1, col=1)
fig.add_vline(x=np.log10(MIN_PEAK + 1), line_dash="dash", line_color="red", row=1, col=2)
fig.update_layout(template="plotly_white", height=420, showlegend=False, title="Distribución del historial y la escala de los juegos")
fig.show()

In [ ]:
FEATURES = [
    "log_peak_players",
    "months_observed",
    "pct_time_to_peak",
    "retention_ratio",
    "trend_slope_yearly",
    "volatility",
    "log_peak_to_avg",
]

games = (
    per_game
    .filter((F.col("months_observed") >= MIN_MONTHS) & (F.col("peak_players") >= MIN_PEAK))
    .withColumn("log_peak_players", F.log1p("peak_players"))
    .withColumn("months_observed", F.col("months_observed").cast("double"))
    .withColumn("pct_time_to_peak", (F.col("peak_idx") - F.col("first_idx")) / F.greatest(F.col("last_idx") - F.col("first_idx"), F.lit(1)))
    .withColumn("retention_ratio", F.expr("try_divide(avg_last3, peak_players)"))
    .withColumn("trend_slope_yearly", F.col("slope_monthly") * 12)
    .withColumn("log_peak_to_avg", F.log(F.greatest(F.col("mean_peak_to_avg"), F.lit(1.0))))
)

# Orden fijo y una sola partición para que K-Means sea reproducible.
games_model = games.na.drop(subset=FEATURES).orderBy("app_id").coalesce(1).cache()
print(f"Juegos que entran al modelo: {games_model.count():,}")

### 4.2 Revisión de las variables

Dos variables con correlación muy alta (|r| > 0,85) dan casi la misma información y pesarían doble en la distancia.

In [ ]:
display(games_model.select(FEATURES).summary("mean", "stddev", "min", "50%", "max"))

correlation = games_model.select(FEATURES).toPandas().corr().round(2)
fig = px.imshow(correlation, text_auto=True, zmin=-1, zmax=1, color_continuous_scale="RdBu_r", title="Correlación entre las variables de ciclo de vida")
fig.update_layout(template="plotly_white", height=560)
fig.show()

high_pairs = [(a, b) for i, a in enumerate(FEATURES) for b in FEATURES[i + 1:] if abs(correlation.loc[a, b]) > 0.85]
print("Pares con |r| > 0,85:", high_pairs or "ninguno")

---
## 5. Preparación: ensamblar y estandarizar

`VectorAssembler` junta las siete variables en un vector y `StandardScaler` las deja con media 0 y desviación 1.

In [ ]:
prep_pipeline = Pipeline(stages=[
    VectorAssembler(inputCols=FEATURES, outputCol="raw_features"),
    StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True),
])
df_prep = prep_pipeline.fit(games_model).transform(games_model).cache()
df_prep.select("app_id", "name", "features").show(3, truncate=100)

---
## 6. ¿Cuántos grupos? Silueta y método del codo

- **Silueta:** mide qué tan bien separado queda cada grupo. Va de −1 a 1 y más alto es mejor.
- **Inercia:** siempre baja al añadir grupos; se busca el punto donde deja de bajar con fuerza (el codo).

Se elige el $k$ con mayor silueta.

In [ ]:
evaluator = ClusteringEvaluator(featuresCol="features", metricName="silhouette", distanceMeasure="squaredEuclidean")

rows = []
for k in K_VALUES:
    model = KMeans(featuresCol="features", k=k, seed=SEED, maxIter=100).fit(df_prep)
    sizes = model.summary.clusterSizes
    rows.append({
        "k": k,
        "silueta": round(evaluator.evaluate(model.transform(df_prep)), 4),
        "inercia": round(model.summary.trainingCost, 1),
        "grupo_mas_pequeno_pct": round(100 * min(sizes) / sum(sizes), 1),
    })

k_results = pd.DataFrame(rows)
display(k_results)

K_FINAL = int(k_results.loc[k_results["silueta"].idxmax(), "k"])
print(f"k elegido: {K_FINAL}")

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=k_results["k"], y=k_results["silueta"], mode="lines+markers", name="Silueta"), secondary_y=False)
fig.add_trace(go.Scatter(x=k_results["k"], y=k_results["inercia"], mode="lines+markers", name="Inercia", line=dict(dash="dot")), secondary_y=True)
fig.add_vline(x=K_FINAL, line_dash="dash", line_color="red", annotation_text=f"k = {K_FINAL}")
fig.update_layout(template="plotly_white", height=460, title="Selección del número de arquetipos")
fig.update_xaxes(title_text="k (número de grupos)", dtick=1)
fig.update_yaxes(title_text="Silueta", secondary_y=False)
fig.update_yaxes(title_text="Inercia", secondary_y=True)
fig.show()

---
## 7. Modelo final

Los números de grupo que entrega K-Means son arbitrarios. Para que siempre signifiquen lo mismo, se renumeran según el tamaño del pico del centroide: el grupo 0 es el de menor pico y el último el de mayor pico. Con $k = 3$ los nombres de negocio se asignaron tras revisar la ficha técnica de la sección 8.

In [ ]:
CLUSTER_NAMES = {0: "Fuegos artificiales", 1: "Gran estreno que se desinfla", 2: "Clásicos de siempre"}

kmeans_model = KMeans(featuresCol="features", k=K_FINAL, seed=SEED, maxIter=100).fit(df_prep)
predictions = kmeans_model.transform(df_prep)
print(f"Silueta final: {evaluator.evaluate(predictions):.4f}")

centers = pd.DataFrame(kmeans_model.clusterCenters(), columns=FEATURES)
new_id = {old: new for new, old in enumerate(centers["log_peak_players"].sort_values().index)}
centers = centers.rename(index=new_id).sort_index()

mapping = spark.createDataFrame(
    [(old, new, CLUSTER_NAMES.get(new, f"Cluster {new}")) for old, new in new_id.items()],
    ["prediction", "cluster_id", "arquetipo"],
)
clustered = predictions.join(mapping, "prediction").drop("prediction").cache()

ARCHETYPES = [CLUSTER_NAMES.get(i, f"Cluster {i}") for i in range(K_FINAL)]
display(clustered.groupBy("cluster_id", "arquetipo").count().orderBy("cluster_id"))

---
## 8. Perfil de cada arquetipo

Cada grupo se describe con sus variables de entrada y con información que **no** se usó para formarlo:

- **Catálogo (`dim_game`):** precio, porcentaje de juegos gratuitos, géneros y reseñas positivas.
- **Twitch (`twitch_steam_relationship`):** si el juego estuvo en el top 200 mensual, cuántos meses y cuántas horas vistas. Solo se comparan juegos con datos dentro de la ventana de Twitch (2016-01 a 2024-09).

In [ ]:
twitch_game = twitch.groupBy("app_id").agg(
    F.count("*").alias("months_in_top200"),
    F.sum("hours_watched").alias("total_hours_watched"),
    F.min("twitch_rank").alias("best_rank"),
)

profile = (
    clustered.select("app_id", "name", "cluster_id", "arquetipo", "peak_players", "peak_idx", "first_month", "last_month", *FEATURES)
    .join(dim_game.select("app_id", "price", "genres", "review_ratio"), "app_id", "left")
    .join(twitch_game, "app_id", "left")
    .fillna({"months_in_top200": 0, "total_hours_watched": 0})
    .withColumn("is_free", (F.col("price") == 0).cast("double"))
    .withColumn("in_top200", (F.col("months_in_top200") > 0).cast("double"))
    .withColumn("in_twitch_window", (F.col("last_month") >= F.lit(TWITCH_START)) & (F.col("first_month") <= F.lit(TWITCH_END)))
    .cache()
)

ficha = (
    profile.groupBy("cluster_id", "arquetipo")
    .agg(
        F.count("*").alias("juegos"),
        F.percentile_approx("peak_players", 0.5).alias("pico_mediano"),
        F.percentile_approx("months_observed", 0.5).alias("meses_medianos"),
        F.round(F.percentile_approx("pct_time_to_peak", 0.5), 2).alias("momento_del_pico"),
        F.round(F.percentile_approx("retention_ratio", 0.5), 3).alias("retencion"),
        F.round(F.percentile_approx("volatility", 0.5), 3).alias("volatilidad"),
        F.round(F.avg("price"), 2).alias("precio_medio"),
        F.round(100 * F.avg("is_free"), 1).alias("pct_gratuitos"),
        F.round(F.avg("review_ratio"), 3).alias("resenas_positivas"),
    )
    .orderBy("cluster_id")
    .toPandas()
)
ficha["pct_juegos"] = (100 * ficha["juegos"] / ficha["juegos"].sum()).round(1)
display(ficha)

In [ ]:
genres = profile.select("cluster_id", "arquetipo", "app_id", F.explode("genres").alias("genre"))
top_genres = [r["genre"] for r in genres.groupBy("genre").count().orderBy(F.desc("count"), "genre").limit(8).collect()]

genre_pd = genres.filter(F.col("genre").isin(top_genres)).groupBy("arquetipo", "genre").count().toPandas()
genre_pd = genre_pd.merge(ficha[["arquetipo", "juegos"]], on="arquetipo")
genre_pd["pct"] = (100 * genre_pd["count"] / genre_pd["juegos"]).round(1)
genre_matrix = genre_pd.pivot(index="arquetipo", columns="genre", values="pct").reindex(index=ARCHETYPES, columns=top_genres).fillna(0)

fig = px.imshow(
    genre_matrix,
    text_auto=".1f",
    color_continuous_scale="Blues",
    aspect="auto",
    title="Composición de géneros por arquetipo (% de juegos del grupo con ese género)",
    labels={"x": "Género", "y": "Arquetipo", "color": "% juegos"},
)
fig.update_layout(template="plotly_white", height=480)
fig.show()

In [ ]:
top_by_peak = Window.partitionBy("cluster_id").orderBy(F.desc("peak_players"), "app_id")
display(
    profile.withColumn("puesto", F.row_number().over(top_by_peak))
    .filter(F.col("puesto") <= 5)
    .select("arquetipo", "puesto", "name", F.round("peak_players", 0).alias("pico_jugadores"), "months_observed", F.round("retention_ratio", 3).alias("retencion"))
    .orderBy("cluster_id", "puesto")
)

### 8.1 Curva típica de cada arquetipo

Es el gráfico central del análisis. Cada juego se alinea respecto al **mes de su pico** (mes 0) y sus jugadores se expresan como % del pico. Luego se toma la mediana por arquetipo. Así se compara la **forma** del ciclo de vida sin que influya el tamaño de cada juego.

In [ ]:
curve = (
    monthly.select("app_id", "month_idx", "avg_players")
    .join(clustered.select("app_id", "cluster_id", "arquetipo", "peak_idx", "peak_players"), "app_id")
    .withColumn("months_from_peak", F.col("month_idx") - F.col("peak_idx"))
    .filter(F.col("months_from_peak").between(-24, 60))
    .withColumn("pct_of_peak", 100 * F.col("avg_players") / F.col("peak_players"))
    .groupBy("cluster_id", "arquetipo", "months_from_peak")
    .agg(F.percentile_approx("pct_of_peak", 0.5).alias("pct_pico"), F.countDistinct("app_id").alias("juegos"))
    .filter(F.col("juegos") >= 20)
    .orderBy("cluster_id", "months_from_peak")
    .toPandas()
)

fig = px.line(
    curve,
    x="months_from_peak",
    y="pct_pico",
    color="arquetipo",
    category_orders={"arquetipo": ARCHETYPES},
    title="Curva típica de jugadores por arquetipo (mediana, % del pico)",
    labels={"months_from_peak": "Meses respecto al pico (0 = mes del pico)", "pct_pico": "% del pico de jugadores", "arquetipo": "Arquetipo"},
    template="plotly_white",
)
fig.add_vline(x=0, line_dash="dot", line_color="gray")
fig.update_layout(height=520)
fig.show()

display(curve[curve["months_from_peak"].isin([-12, -6, 0, 6, 12, 24, 36, 60])].pivot(index="arquetipo", columns="months_from_peak", values="pct_pico").round(1).reset_index())

### 8.2 Centroides y mapa PCA

- **Mapa de calor de centroides:** cuántas desviaciones estándar se aleja cada grupo del promedio en cada variable. Rojo = por encima; azul = por debajo.
- **PCA 2D:** resume las siete variables en dos ejes para ver cómo se separan los grupos. Es una proyección, así que parte de la separación real no se ve.

In [ ]:
heat = centers.round(2)
heat.index = ARCHETYPES
fig = px.imshow(
    heat,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-2,
    zmax=2,
    aspect="auto",
    title="Centroides estandarizados (desviaciones respecto al promedio)",
    labels={"x": "Variable", "y": "Arquetipo", "color": "z"},
)
fig.update_layout(template="plotly_white", height=460)
fig.show()

In [ ]:
pca_model = PCA(k=2, inputCol="features", outputCol="pca").fit(clustered)
explained = pca_model.explainedVariance.toArray()

pca_pd = (
    pca_model.transform(clustered)
    .select("name", "arquetipo", "peak_players", vector_to_array("pca")[0].alias("PC1"), vector_to_array("pca")[1].alias("PC2"))
    .toPandas()
)

fig = px.scatter(
    pca_pd,
    x="PC1",
    y="PC2",
    color="arquetipo",
    category_orders={"arquetipo": ARCHETYPES},
    hover_data=["name", "peak_players"],
    opacity=0.6,
    title=f"Mapa 2D de los arquetipos (PCA: {explained[0]:.0%} + {explained[1]:.0%} de la varianza)",
    template="plotly_white",
)
fig.update_layout(height=600, legend_title_text="Arquetipo")
fig.show()

### 8.3 Exposición en Twitch por arquetipo

La comparación es **descriptiva**: si un arquetipo concentra más exposición en Twitch hay una asociación, pero no prueba que Twitch cause ese ciclo de vida. También puede ocurrir lo contrario: que los juegos con comunidades grandes y estables atraigan más streams.

In [ ]:
twitch_by_cluster = (
    profile.filter("in_twitch_window")
    .groupBy("cluster_id", "arquetipo")
    .agg(
        F.count("*").alias("juegos"),
        F.round(100 * F.avg("in_top200"), 1).alias("pct_alguna_vez_top200"),
        F.round(F.avg("months_in_top200"), 2).alias("meses_top200_promedio"),
        F.percentile_approx(F.when(F.col("in_top200") == 1, F.col("total_hours_watched")), 0.5).alias("horas_vistas_medianas"),
        F.min("best_rank").alias("mejor_puesto"),
    )
    .orderBy("cluster_id")
    .toPandas()
)
display(twitch_by_cluster)

fig = make_subplots(rows=1, cols=2, subplot_titles=("% de juegos que alguna vez estuvo en el top 200", "Meses promedio en el top 200"))
fig.add_trace(go.Bar(x=twitch_by_cluster["arquetipo"], y=twitch_by_cluster["pct_alguna_vez_top200"], text=twitch_by_cluster["pct_alguna_vez_top200"]), row=1, col=1)
fig.add_trace(go.Bar(x=twitch_by_cluster["arquetipo"], y=twitch_by_cluster["meses_top200_promedio"], text=twitch_by_cluster["meses_top200_promedio"]), row=1, col=2)
fig.update_layout(template="plotly_white", height=460, showlegend=False, title="Exposición en Twitch por arquetipo")
fig.show()

---
## 9. Interpretación de resultados

### Calidad de la segmentación
- Se eligió **k = 3** porque tiene la mayor silueta (0,48), un 50 % más que k = 4 (0,32). La inercia baja de forma suave y no muestra un codo claro, así que la decisión la marca la silueta.
- Una silueta de 0,48 indica grupos **razonablemente separados**: se distinguen bien, aunque algunos juegos de los bordes podrían ir en dos grupos.

### Arquetipos encontrados
- **Gran estreno que se desinfla (66 % de los juegos):** el pico llega al inicio de su vida y luego los jugadores bajan poco a poco hasta cerca del 10 % del pico. Es el grupo con más juegos de rol. Ejemplos: PUBG, HELLDIVERS 2, Monster Hunter: World.
- **Clásicos de siempre (23 %):** el pico llega tarde y conservan cerca del 45 % de los jugadores años después. Son los mejor valorados (86 % de reseñas positivas) y tienen más estrategia y simulación. Ejemplos: Counter-Strike 2, GTA V, Rust, Terraria.
- **Fuegos artificiales (12 %):** un único mes gigante, unas 17 veces mayor que un mes normal, y muy alta volatilidad. Son sobre todo indie (82 %) y casuales (45 %), baratos (unos 6 USD) y con peores reseñas (70 %).

### Exposición en Twitch
- Los clásicos pasan en promedio 3,8 meses en el top 200, casi cinco veces más que los juegos de estreno (0,8), y cuando aparecen acumulan más del doble de horas vistas. Los fuegos artificiales casi nunca aparecen.
- Es una **asociación, no una causa**: no se sabe si Twitch hace durar al juego o si el juego, por ser duradero, atrae a los streamers. El Cuaderno ML 2 pone esto a prueba.

### Aplicaciones en el proyecto
1. **Segmentación del catálogo:** comparar un juego con otros de su mismo arquetipo y no con el promedio de Steam.
2. **Señal de alerta:** un juego cuya curva se parece a la de un fuego artificial puede priorizarse para acciones de retención.

### Limitaciones
- Los juegos lanzados antes de 2012 se observan desde la mitad de su vida, así que su curva está incompleta.
- Las reseñas son una foto actual; sirven para describir los grupos, no para explicarlos.
- El cruce con Twitch usa solo nombres con coincidencia única; los juegos con nombre ambiguo cuentan como "sin exposición".